# LSTM Regression — Jena Climate Dataset
**Target:** `T (degC)` · **Builds on:** `output/processed_data.csv` (Stage 1) + `scaler.pkl` (Stage 2)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from utils.data_prep import load_and_split, scale_features
from utils.sequencer import make_sequences
from utils.models import build_lstm_1layer, build_lstm_2layer
from utils.trainer import compile_and_fit
from utils.evaluator import (
    inverse_transform_target,
    compute_metrics,
    metrics_table,
    plot_loss_curves,
    plot_actual_vs_predicted,
    plot_window_experiment,
)

DATA_PATH   = Path('../output/processed_data.csv')
SCALER_PATH = Path('../output/scaler.pkl')       # Stage-2 feature scaler — never refit
LSTM_PATH   = Path('../output/lstm_best.h5')

# Stage-2 model paths for cross-stage comparison
STAGE2_MLP_PATH = Path('../../assignment2/output/mlp_best.keras')
STAGE2_DNN_PATH = Path('../../assignment2/output/dnn_best.keras')

DEFAULT_WINDOW = 24   # 24 h of hourly data

print('TF version:', tf.__version__)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

---
## Section 1 — Data Preparation & Sequence Construction

In [ ]:
train_df, val_df, test_df = load_and_split(DATA_PATH)

In [ ]:
# Reload Stage-2 scaler; scale features + target (target scaled separately)
X_tr, y_tr, X_va, y_va, X_te, y_te, target_scaler, _ = scale_features(
    train_df, val_df, test_df, scaler_path=SCALER_PATH
)
N_FEATURES = X_tr.shape[1]
print(f'n_features: {N_FEATURES}')

In [ ]:
# Build sequences with the default window (applied separately to each split)
X_train_seq, y_train_seq = make_sequences(X_tr, y_tr, DEFAULT_WINDOW)
X_val_seq,   y_val_seq   = make_sequences(X_va, y_va, DEFAULT_WINDOW)
X_test_seq,  y_test_seq  = make_sequences(X_te, y_te, DEFAULT_WINDOW)

print(f'X_train_seq: {X_train_seq.shape}  y_train_seq: {y_train_seq.shape}')
print(f'X_val_seq  : {X_val_seq.shape}  y_val_seq  : {y_val_seq.shape}')
print(f'X_test_seq : {X_test_seq.shape}  y_test_seq : {y_test_seq.shape}')

---
## Section 2 — LSTM Models

### 2a — Single-layer LSTM (baseline)

In [ ]:
lstm_1l = build_lstm_1layer(DEFAULT_WINDOW, N_FEATURES)
lstm_1l.summary()

In [ ]:
history_1l = compile_and_fit(
    lstm_1l, X_train_seq, y_train_seq, X_val_seq, y_val_seq,
    checkpoint_path=Path('../output/lstm_1l_best.h5'),
    epochs=200, batch_size=64, patience=10,
)

### 2b — Two-layer stacked LSTM (main model)

In [ ]:
lstm_2l = build_lstm_2layer(DEFAULT_WINDOW, N_FEATURES)
lstm_2l.summary()

In [ ]:
history_2l = compile_and_fit(
    lstm_2l, X_train_seq, y_train_seq, X_val_seq, y_val_seq,
    checkpoint_path=LSTM_PATH,
    epochs=200, batch_size=64, patience=10,
)

In [ ]:
plot_loss_curves({"LSTM_1L": history_1l, "LSTM_2L": history_2l})

---
## Section 3 — Evaluation & Cross-Stage Comparison

In [ ]:
splits_seq = {
    "train": (X_train_seq, y_train_seq),
    "val":   (X_val_seq,   y_val_seq),
    "test":  (X_test_seq,  y_test_seq),
}

metrics_1l = compute_metrics(lstm_1l, splits_seq, target_scaler)
metrics_2l = compute_metrics(lstm_2l, splits_seq, target_scaler)

print('LSTM 1-layer:', metrics_1l)
print('LSTM 2-layer:', metrics_2l)

In [ ]:
# ── Pull Stage-2 metrics for comparison ──────────────────────────────────────
# Stage-2 used unscaled target, so we compute metrics on the flat (non-sequence)
# test split directly from the Stage-2 checkpoints.

import pickle

# Reload Stage-2 feature scaler to reconstruct the Stage-2 test arrays
with open(SCALER_PATH, 'rb') as f:
    feat_scaler = pickle.load(f)

_drop = ['time_of_day', 'T (degC)']
_feat_cols = [c for c in train_df.columns if c not in _drop]

X_te_s2 = feat_scaler.transform(test_df[_feat_cols].values)
y_te_s2  = test_df['T (degC)'].values

# compile=False avoids KerasSaveable deserialization errors for string metrics
mlp_s2 = tf.keras.models.load_model(STAGE2_MLP_PATH, compile=False)
dnn_s2 = tf.keras.models.load_model(STAGE2_DNN_PATH, compile=False)

def _s2_metrics(model, X, y):
    p = model.predict(X, verbose=0).flatten()
    return {
        'test': {
            'RMSE': float(np.sqrt(np.mean((y - p)**2))),
            'MAE':  float(np.mean(np.abs(y - p))),
        }
    }

mlp_metrics_s2 = _s2_metrics(mlp_s2, X_te_s2, y_te_s2)
dnn_metrics_s2 = _s2_metrics(dnn_s2, X_te_s2, y_te_s2)

print('MLP (Stage 2):', mlp_metrics_s2)
print('DNN (Stage 2):', dnn_metrics_s2)

In [ ]:
table = metrics_table(mlp_metrics_s2, dnn_metrics_s2, metrics_1l, metrics_2l)
print(table.to_string())
table

In [ ]:
# Actual vs Predicted — best LSTM (2-layer)
plot_actual_vs_predicted(lstm_2l, X_test_seq, y_test_seq, target_scaler, model_name='LSTM_2L', n=500)

### Conclusion — Does LSTM improve over MLP/DNN?

**Why LSTM can outperform flat models on this task:**  
MLP and DNN receive a hand-crafted feature vector (lag_1h, lag_2h, lag_3h, rolling means) that encodes recent history only up to 3 hours. The LSTM receives a raw sliding window of the last 24 hours of all sensor readings and learns its own temporal filters end-to-end. This allows it to capture:
- longer-range dependencies (diurnal cycle, weather-front signatures)
- non-linear interactions across the full window that fixed-lag features miss

**Why the improvement may be modest:**  
The lag/rolling features in Stage 1 already encode most short-term autocorrelation. The MLP/DNN then only need to learn a nearly-linear mapping from those features, which they do very efficiently. The LSTM adds recurrence but must learn from scratch what the engineered features express explicitly.

**Two-layer vs one-layer:**  
The second LSTM layer allows the network to learn higher-order temporal patterns (e.g. how the rate-of-change of temperature itself evolves), at the cost of more parameters and longer training.

---
## Section 4 — Window Size Experiment

In [ ]:
WINDOW_SIZES = [12, 24, 48]
window_results = {}   # window_size → {RMSE, MAE}

for ws in WINDOW_SIZES:
    print(f'\n=== Window size: {ws} ===')
    Xtr_w, ytr_w = make_sequences(X_tr, y_tr, ws)
    Xva_w, yva_w = make_sequences(X_va, y_va, ws)
    Xte_w, yte_w = make_sequences(X_te, y_te, ws)

    m = build_lstm_2layer(ws, N_FEATURES)
    compile_and_fit(
        m, Xtr_w, ytr_w, Xva_w, yva_w,
        checkpoint_path=Path(f'../output/lstm_window_{ws}.h5'),
        epochs=200, batch_size=64, patience=10, verbose=0,
    )

    y_pred_sc = m.predict(Xte_w, verbose=0).flatten()
    y_true_c  = inverse_transform_target(yte_w, target_scaler)
    y_pred_c  = inverse_transform_target(y_pred_sc, target_scaler)
    rmse = float(np.sqrt(np.mean((y_true_c - y_pred_c)**2)))
    mae  = float(np.mean(np.abs(y_true_c - y_pred_c)))
    window_results[ws] = {'RMSE': rmse, 'MAE': mae}
    print(f'  RMSE={rmse:.4f} °C  MAE={mae:.4f} °C')

print('\nWindow experiment complete.')

In [ ]:
# Results table
win_df = pd.DataFrame([
    {'Window': ws, 'RMSE (°C)': v['RMSE'], 'MAE (°C)': v['MAE']}
    for ws, v in window_results.items()
]).set_index('Window')
print(win_df.to_string())
win_df

In [ ]:
plot_window_experiment(
    list(window_results.keys()),
    [v['RMSE'] for v in window_results.values()],
    [v['MAE']  for v in window_results.values()],
)

### Window size conclusion

- **Window = 12 (12 h):** Covers half a diurnal cycle; the model may miss the overnight cooling trend that informs next-morning temperature.
- **Window = 24 (24 h):** Covers one full diurnal cycle — typically the sweet spot for hourly temperature forecasting.
- **Window = 48 (48 h):** Covers two diurnal cycles. Longer windows add more context but also more noise and longer training time; RMSE may improve marginally or plateau.

---
## Section 5 — Save & Verify Artifacts

In [ ]:
# Confirm checkpoint files exist
for p in [
    Path('../output/lstm_1l_best.h5'),
    LSTM_PATH,
    SCALER_PATH,
]:
    size = p.stat().st_size if p.exists() else -1
    status = f'OK  {size:,} bytes' if size >= 0 else 'MISSING'
    print(f'{p.name}: {status}')

In [ ]:
# Reload lstm_best.h5, run one forward pass, assert shape and print official metrics
lstm_reloaded = tf.keras.models.load_model(LSTM_PATH, compile=False)

y_pred_sc = lstm_reloaded.predict(X_test_seq, verbose=0).flatten()
assert y_pred_sc.shape == y_test_seq.shape, \
    f'Shape mismatch: {y_pred_sc.shape} vs {y_test_seq.shape}'

y_true_c = inverse_transform_target(y_test_seq, target_scaler)
y_pred_c = inverse_transform_target(y_pred_sc, target_scaler)

official_rmse = float(np.sqrt(np.mean((y_true_c - y_pred_c)**2)))
official_mae  = float(np.mean(np.abs(y_true_c - y_pred_c)))

print(f'Official Test RMSE : {official_rmse:.4f} °C')
print(f'Official Test MAE  : {official_mae:.4f} °C')